# infinite_loops
Infinite loops are not just beginner mistakes. In real systems they show up as stuck workers, runaway retryers, hot CPU processes, blocked deployments, and services that never release resources. Understanding how they happen is part of writing safe control flow.

## 1. Problem First
Why does this matter in real systems?
- A worker may keep retrying forever and spam an external API.
- A polling loop may burn CPU while waiting for state that never changes.
- A service loop without shutdown logic can block clean termination.

In [1]:
counter = 3
steps = 0
while counter > 0 and steps < 4:
    print(counter)
    steps += 1

3
3
3
3


## 2. Minimal Concept
An infinite loop happens when a loop condition never becomes false or the loop never hits a terminating `break`.
- `while True:` is intentionally infinite unless something exits it.
- A loop can also be accidentally infinite if state never changes.
- Infinite does not always mean obvious; some loops are logically infinite only under certain input states.

## 3. Mental Model
How Python actually behaves
Python does exactly what the loop condition tells it. If the condition is always truthy, or if the loop body fails to move state toward termination, Python will continue forever. The interpreter does not infer your intended stop condition.

In [2]:
flag = True
iterations = 0
while flag and iterations < 3:
    print("still running")
    iterations += 1

still running
still running
still running


## 4. Internal Mechanics
An infinite loop is just repeated conditional jumping with no path to exit. In worker processes and servers, intentionally infinite loops are common, but they must include safe exit conditions, sleeps, backoff, or signal handling.

In [3]:
import dis

def loop_forever(limit):
    while True:
        if limit <= 0:
            break
        limit -= 1
    return limit

dis.dis(loop_forever)
print(loop_forever(2))

  3           0 RESUME                   0

  4           2 NOP

  5     >>    4 LOAD_FAST                0 (limit)
              6 LOAD_CONST               1 (0)
              8 COMPARE_OP              26 (<=)
             12 POP_JUMP_IF_FALSE        3 (to 20)

  6          14 NOP

  8          16 LOAD_FAST                0 (limit)
             18 RETURN_VALUE

  7     >>   20 LOAD_FAST                0 (limit)
             22 LOAD_CONST               2 (1)
             24 BINARY_OP               23 (-=)
             28 STORE_FAST               0 (limit)

  4          30 JUMP_BACKWARD           14 (to 4)
0


## 5. Edge Cases
Where it breaks:
- The wrong variable gets updated, so progress never happens.
- External state is assumed to change, but it never does.
- A `continue` can skip the code that would have terminated the loop.
- Retry loops can become logically infinite even with changing counters if reset rules are wrong.

In [4]:
attempt = 0
safety_steps = 0
while attempt < 3 and safety_steps < 5:
    print(f"attempt={attempt}")
    safety_steps += 1
print("attempt never changed")

attempt=0
attempt=0
attempt=0
attempt=0
attempt=0
attempt never changed


## 6. Performance Thinking
- Infinite loops can drive CPU to 100% if they spin without waiting.
- Polling loops should often sleep or back off.
- Runaway loops can multiply downstream cost by hammering APIs, queues, or databases.
- A bounded loop is usually easier to reason about than an open-ended one.

## 7. Real Use Case
A background worker may run forever by design, but it still needs controlled exit and non-busy waiting.

In [5]:
jobs = ["job1", "job2"]
processed = 0
while True:
    if processed >= len(jobs):
        break
    print(f"processing {jobs[processed]}")
    processed += 1

processing job1
processing job2


## 8. Anti-Pattern
What beginners do wrong:
- Write `while True` without planning the exit path first.
- Depend on external systems to eventually change without a timeout.
- Forget sleeps or backoff in retry and polling loops.

In [6]:
retries = 0
max_retries = 3
while True:
    print("retrying")
    retries += 1
    if retries >= max_retries:
        break

retrying
retrying
retrying


## 9. Interview Signals
Questions FAANG asks:
- How do infinite loops happen in real code, not just toy examples?
- When is `while True` acceptable?
- What safeguards belong in retry and polling loops?
- How can `continue` accidentally create a non-terminating loop?

## 10. Exercise (Non-trivial)
Design a worker loop that polls a queue indefinitely, processes jobs when available, sleeps when the queue is empty, exits on shutdown, and stops retrying poison jobs after a bounded number of failures. Explain how each path prevents runaway behavior.

In [7]:
def worker_loop(shutdown, jobs):
    index = 0
    while not shutdown:
        if index >= len(jobs):
            continue
        print(jobs[index])
        index += 1

# Task:
# 1. Find the busy-loop bug.
# 2. Add a safe idle path.
# 3. Add bounded retry and shutdown handling.